In [ ]:
import asyncio
import json
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from dotenv import load_dotenv
load_dotenv(Path('..') / '.env')

from agents import Agent, Runner
from agents.mcp import MCPServerStreamableHttp
from agents.stream_events import RunItemStreamEvent

AIRTABLE_MCP_URL = 'https://mcp.airtable.com/mcp'
token = os.environ.get('AIRTABLE_TOKEN', '')
print(f'Token loaded: {bool(token)}')

In [ ]:
def make_server():
    return MCPServerStreamableHttp(
        params={
            'url': AIRTABLE_MCP_URL,
            'headers': {'Authorization': f'Bearer {token}'},
        },
        cache_tools_list=True,
    )

INSTRUCTIONS = (
    'You are a helpful assistant with access to Airtable. '
    'Always call the necessary tools immediately and include the results in your response. '
    'Complete tasks fully before responding.'
)

In [ ]:
async def run_traced(prompt: str):
    """Run the agent and print every tool call and output as it happens."""
    async with make_server() as server:
        agent = Agent(name='Airtable Agent', instructions=INSTRUCTIONS, mcp_servers=[server])
        stream = Runner.run_streamed(agent, prompt, max_turns=10)
        async for event in stream.stream_events():
            if isinstance(event, RunItemStreamEvent):
                if event.name == 'tool_called':
                    item = event.item
                    print(f'\n→ TOOL CALL: {item.raw_item.name}')
                    try:
                        args = json.loads(item.raw_item.arguments)
                        print(f'  args: {json.dumps(args, indent=2)}')
                    except Exception:
                        print(f'  args: {item.raw_item.arguments}')
                elif event.name == 'tool_output':
                    item = event.item
                    output = item.output if hasattr(item, 'output') else str(item)
                    try:
                        parsed = json.loads(output) if isinstance(output, str) else output
                        print(f'← OUTPUT: {json.dumps(parsed, indent=2)[:800]}')
                    except Exception:
                        print(f'← OUTPUT: {str(output)[:800]}')
        print(f'\n=== FINAL ANSWER ===\n{stream.final_output_as(str)}')
        return stream

In [ ]:
# Change this prompt to test different interactions
PROMPT = 'Which bases do I have available?'

result = await run_traced(PROMPT)